In [1]:
from models.gpkg import GeoPackage

In [2]:
gpkg = GeoPackage()

In [3]:
from pprint import pprint
sigmine_gdf = gpkg.read_layer('sigmine_rs')

c:\Users\adminitsd\Documents\ufrgs\mestrado\python_god\venv\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


In [4]:
gpkg.layer_list

['pampa',
 'faixa_transicao',
 'ti_ac_preliminar',
 'sistemas_ecologicos_banda_oriental',
 'veg_biomas_a',
 'ti_inside_pampa',
 'ti_outside_pampa',
 'ti_inside_pampa_lin',
 'ti_inside_pampa_aoi',
 'ti_outside_pampa_lin',
 'ti_outside_pampa_aoi',
 'se_rs_clip',
 'mancha_sentinelf',
 'uc_br',
 'sigmine_rs',
 'rs_aoi_polyconic',
 'rs_buffer_poly']

In [5]:
ti_gdf = gpkg.read_layer('ti_ac_preliminar')


In [7]:
ti_gdf.head()

,modfiscais,area,inscricao,Name,area_ha,other_name,population,land_tenur,cd_setor,nm_rgi,...,Shape_Le_2,Shape_Area,cob_pc,p_lpi,p_ed,p_frac,cap_adap,ca_norm,source,geometry
0,0.0,0.0,NaT,Polígono sem título,296.810510,None,None,None,None,None,...,8412.584106,2.973008e+06,88.375408,0.000176,0.002544,0.000390,0.003110,0.004965,None,"MULTIPOLYGON Z (((5366397.945 6711242.732 0, 5..."
1,0.0,0.0,NaT,None,329.197196,None,None,None,431490205001288P,Porto Alegre,...,8655.569930,3.295298e+06,98.142809,0.000000,0.000000,0.000000,0.000000,0.000000,IBGE Setores Censo 2022,"MULTIPOLYGON Z (((5287377.789 6653737.619 0, 5..."
2,0.0,0.0,NaT,None,14.587715,None,None,None,431490205002552P,Porto Alegre,...,1943.871532,1.460147e+05,58.086403,0.000000,0.000000,0.000000,0.000000,0.000000,IBGE Setores Censo 2022,"MULTIPOLYGON Z (((5276505.692 6670909.123 0, 5..."
3,0.0,0.0,NaT,None,113.033699,None,None,None,None,None,...,7056.625671,1.131268e+06,93.953685,0.000000,0.000000,0.000000,0.000000,0.000000,DNIT,"MULTIPOLYGON Z (((5259479.817 6647130.844 0, 5..."
4,0.0,0.0,2018-05-30,None,55.585153,None,None,None,None,None,...,3795.971040,5.568025e+05,80.441897,0.000941,0.005745,0.001117,0.007803,0.012456,None,"MULTIPOLYGON Z (((5372998.368 6713607.226 0, 5..."


In [8]:
import numpy as np

future_condition = [
'REQUERIMENTO DE PESQUISA',
'REQUERIMENTO DE LAVRA GARIMPEIRA',
'REQUERIMENTO DE REGISTRO DE EXTRAÇÃO',
'REQUERIMENTO DE LICENCIAMENTO',
'DIREITO DE REQUERER A LAVRA',
'DADO NÃO CADASTRADO',
'AUTORIZAÇÃO DE PESQUISA',
'DISPONIBILIDADE',
'APTO PARA DISPONIBILIDADE'
]

condition = sigmine_gdf['fase'].isin(future_condition)

sigmine_gdf['status_proj'] = np.where(condition, 'futuro', 'corrente')
sigmine_gdf['status_weight'] = np.where(sigmine_gdf['status_proj'].isin(['futuro']), 0.5, 1)

In [9]:
sigmine_gdf.head()

,id,processo,numero,ano,area_ha,fase,ult_evento,nome,subs,uso,uf,dsprocesso,geometry,status_proj,status_weight
0,1,2839/1935,2839,1935,2346.20,CONCESSÃO DE LAVRA,418 - CONC LAV/RAL ANO BASE APRESENTADO EM 13/...,COPELMI MINERAÇÃO LTDA,ARGILA,Não informado,RS,002.839/1935,"MULTIPOLYGON Z (((5189565.419 6662471.583 0, 5...",corrente,1.0
1,2,2122/1936,2122,1936,1778.30,CONCESSÃO DE LAVRA,418 - CONC LAV/RAL ANO BASE APRESENTADO EM 11/...,Luzia Jurema Vidal de Souza,CHUMBO,Não informado,RS,002.122/1936,"MULTIPOLYGON Z (((5009177.515 6589925.229 0, 5...",corrente,1.0
2,3,212201/1936,212201,1936,1064.87,CONCESSÃO DE LAVRA,418 - CONC LAV/RAL ANO BASE APRESENTADO EM 11/...,Luzia Jurema Vidal de Souza,CHUMBO,Não informado,RS,212.201/1936,"MULTIPOLYGON Z (((5015186.591 6587903.658 0, 5...",corrente,1.0
3,4,4632/1938,4632,1938,98.01,CONCESSÃO DE LAVRA,436 - CONC LAV/DOCUMENTO DIVERSO PROTOCOLIZADO...,COMICAN - COMPANHIA DE MINERACAO CANDIOTA,CALCÁRIO,Não informado,RS,004.632/1938,"MULTIPOLYGON Z (((5100875.815 6454192.067 0, 5...",corrente,1.0
4,5,2942/1939,2942,1939,1083.84,CONCESSÃO DE LAVRA,470 - CONC LAV/EXIGÊNCIA PUBLICADA EM 20/05/2024,COMPANHIA RIOGRANDENSE DE MINERACAO CRM,CARVÃO,Não informado,RS,002.942/1939,"MULTIPOLYGON Z (((5188632.753 6664249.583 0, 5...",corrente,1.0


In [10]:
from geopandas import sjoin
intersect_gdf = sjoin(ti_gdf, sigmine_gdf, how='inner', predicate='intersects')


In [12]:
intersect_gdf['count'] = 1
qt_projetos_aoi_ti = intersect_gdf.groupby('status_proj')['count'].sum().reset_index()

qt_proj_por_aoi_ti = intersect_gdf.groupby('geometry')['count'].sum().reset_index()


In [13]:
qt_proj_por_aoi_ti

,geometry,count
0,"MULTIPOLYGON Z (((4996538.658 6469523.26 0, 49...",13
1,"MULTIPOLYGON Z (((4995382.674 6529423.008 0, 4...",1
2,"MULTIPOLYGON Z (((5161144.233 6457106.873 0, 5...",1
3,"MULTIPOLYGON Z (((5161469.513 6457618.408 0, 5...",1
4,"MULTIPOLYGON Z (((5189766.581 6568524.44 0, 51...",5
5,"MULTIPOLYGON Z (((5183959.249 6584678.11 0, 51...",4
6,"MULTIPOLYGON Z (((5033875.358 6675688.42 0, 50...",3
7,"MULTIPOLYGON Z (((5028811.043 6711407.748 0, 5...",1
8,"MULTIPOLYGON Z (((5076981.62 6777674.434 0, 50...",2
9,"MULTIPOLYGON Z (((5075315.51 6991996.201 0, 50...",2


In [23]:
geoms = [feature["geometry"] for feature in gdf.__geo_interface__["features"]]

In [26]:
from rasterio.mask import mask

clip, transform = mask(mp_2022, geoms, crop=True, all_touched=True, filled=False)

In [28]:
out_meta = mp_2022.meta.copy()
out_meta.update({
    "height": clip.shape[1],
    "width": clip.shape[2],
    "transform": transform
})

In [29]:
import rasterio
with rasterio.open('mapiomas_2022_rs.tif', 'w', **out_meta) as dest:
    dest.write(clip)